In [1]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import relationship, sessionmaker, declarative_base

# 1. 엔진 및 세션 설정
# SQLite를 사용하며, 파일명은 crime_data.db로 설정합니다.
engine = create_engine('sqlite:///crime_data.db', echo=False)
Session = sessionmaker(bind=engine)
session = Session()
Base = declarative_base()

# --- [ 마스터 테이블 ] ---

class RegionMaster(Base):
    """지역 마스터 (예: 서울 강남구, 서울 종로구)"""
    __tablename__ = 'region_master'
    id = Column(Integer, primary_key=True)
    region_name = Column(String(50), unique=True, nullable=False)

    # 관계 설정
    mappers = relationship("RegionMapper", back_populates="region")
    region_crimes = relationship("CrimeRegion", back_populates="region")

class CrimeCategory(Base):
    """범죄 종류 마스터 (예: 절도, 폭행)"""
    __tablename__ = 'crime_category'
    id = Column(Integer, primary_key=True)
    main_cat = Column(String(50))  # 범죄대분류
    sub_cat = Column(String(50), unique=True) # 범죄중분류

    # 관계 설정
    region_stats = relationship("CrimeRegion", back_populates="category")
    time_stats = relationship("CrimeTime", back_populates="category")
    week_stats = relationship("CrimeWeek", back_populates="category")

# --- [ 매핑 테이블 (핵심 연결 고리) ] ---

class RegionMapper(Base):
    """핫스팟과 지역 마스터를 잇는 중간 다리"""
    __tablename__ = 'region_mapper'
    id = Column(Integer, primary_key=True)
    
    # FK 설정
    hotspot_id = Column(Integer, ForeignKey('hotspot_api.id'))
    region_id = Column(Integer, ForeignKey('region_master.id'))

    # 관계 설정
    hotspot = relationship("HotspotAPI", back_populates="mapper")
    region = relationship("RegionMaster", back_populates="mappers")

# --- [ 데이터 테이블 ] ---

class HotspotAPI(Base):
    """핫스팟 API 정보 (mapping_fix 기반)"""
    __tablename__ = 'hotspot_api'
    id = Column(Integer, primary_key=True)
    hotspot_name = Column(String(100), unique=True) # AREA_NM
    area_code = Column(String(20))                  # AREA_CD
    
    # 매핑 테이블과 1:1 연결
    mapper = relationship("RegionMapper", back_populates="hotspot", uselist=False)

class CrimeRegion(Base):
    """경찰청 지역별 범죄 통계 (police_region_fix 기반)"""
    __tablename__ = 'crime_region'
    id = Column(Integer, primary_key=True)
    crime_count = Column(Integer)
    
    region_id = Column(Integer, ForeignKey('region_master.id'))
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    
    region = relationship("RegionMaster", back_populates="region_crimes")
    category = relationship("CrimeCategory", back_populates="region_stats")

class CrimeTime(Base):
    """시간대별 범죄 통계 (전국 통계용)"""
    __tablename__ = 'crime_time'
    id = Column(Integer, primary_key=True)
    time_range = Column(String(30))
    crime_count = Column(Float)
    
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    category = relationship("CrimeCategory", back_populates="time_stats")

class CrimeWeek(Base):
    """요일별 범죄 통계 (전국 통계용)"""
    __tablename__ = 'crime_week'
    id = Column(Integer, primary_key=True)
    day_of_week = Column(String(10))
    crime_count = Column(Integer)
    
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    category = relationship("CrimeCategory", back_populates="week_stats")

# 2. 테이블 생성
Base.metadata.create_all(engine)
print("의도하신 매핑 구조로 DB 생성 완료!")


의도하신 매핑 구조로 DB 생성 완료!
